# Idealista 4: Recuperación de información de los anuncios usando Fireworks AI.

Para poder extraer información de los anuncios, y en particular la información de los comentarios de los anunciantes, donde está la mayor parte de la misma, y de forma desestructurada, que queremos recuperar, construiremos una serie de funciones con las cuales poder conectarnos a los servicios de IA de **Fireworks AI** y extraer la información deseada lanzando un promt con información de cada uno de los anuncios y solicitando un json esturcturado del cual podemos contruir y recuperar información que me ha parecido relevante para el análisis que haremos sobre estos datos.

En este caso en vez de trabajar con funciones asincronas usaremos **pandarallel** para aplicar la extracción de información en paralelo. Aunque ya hemos trabajado con funciones asincronas en los pasos anteriores, esta también es una fórmula con la cual simplificar un poco la redacción del código, paralelizar el trabajo y aprovechar la aplicación de funciones sobre columnas de un dataFrame de **Pandas** de forma directa. Hemos venido usando **PySpark** para trabajar con las DeltaTables donde tenemos los datos, además de estar más familiarizado con este lenguaje es mejor que **Pandas** a nivel computacional por paralelizar los diferentes cálculos y que se realizan sobre las columnas y además, trabaja con DAG, creando un grafo, o plan de ejecución, adecuado para llevar a cabo todas las operaciones definidas por código de la mejor manera para que el coste sea el menor, por ejemplo, si hacemos joins y luego filtros, dependiendo de la naturaleza de los mismos, si es conveniente, Spark lo que hace es aplicar primero el filto y luego el join, de esta forma en el join hay menos datos que cruzar y los recursos utilizados son menores que de lo otra forma aún habiendo redactado el código en ese orden. Volviendo a este trabajo, la idea es seleccionar una serie de columnas de la tabla anterior, construir un json con ellas, y así mismo, con esta, construir una nueva columna en la que guardar el promp con el que queremos que recuperar un json con una serie de campos definidos (hasta ahora todo en PySpark) y luego, pasando a un dataFrame de Pandas, pasar ese promp como entrada a esa función con la cual obtener la respuesta del modelo IA esto lo haremos con un *.apply()* sobre la columna anterior, volver a pasar a un dataFrame de PySpark y aplanar ese json, dejando la información obetnida como difertes columnas que usaremos después para el análisis que queremos llevar a cabo.

La cosa es que Pandas ejecuta las cosas de forma secuencial, un registro tras otro, lo que puede suponer una gran cantidad de timpo, para eso usamos **pandarallel**, nos permite hacer estas operaciones ejecutando en paralelo los diferentes registros, pudiendo llevar a cabo operaciones en los diferentes nodos, replicando de alguna forma lo que hace PySpark, la cuestión de hacer esto con Pandas y no con Spark es que, pese ha imitar el comportamiento distribuido Pandas y Pandarallel se ejecutan en el mismo entorno de Python en el que trabajamos, sin embargo PySpark lleva a cabo las operaciones en el entorno en de sus ejecutores, que están aislados del entorno en el que se encuentra el notebook, realizar el trabajo en PySpark implicaría llevarse todas las funciones, en formato UDF, variables y dependencias al entorno se Spark, lo que complicaría el trabajo.

In [ ]:
import os
import sys
import time
import json
from typing import Dict, Any
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
import pandas as pd
#Para "paralelizar" procesos de Pandas
from pandarallel import pandarallel

sys.path.insert(0, '/tfm/python_notebooks')
from tfm_lib import utils as tfm_utils
from tfm_lib.fireworks_ai.client import query_json_to_fireworks_ai

#Inicializamos pandarallerl, indicamos que vamos a trabajar con los 8 núcleos del contenedor
pandarallel.initialize(nb_workers=8)

INFO: Pandarallel will run on 8 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.


In [2]:
DETALLE_TABLE = 'raw.idealista.detalle_propiedades'
detalle_path = tfm_utils.full_table_path(DETALLE_TABLE)

## 0. Creación de funciones auxiliares para extraer datos de los anuncios
Empezamos con la creación de una función para crear el prompt a partir de una serie de columnas que han sido agrupadas en una sola, en formato con estructura json y además, devolver el resultado solicitado al modelo que seleccionamos en FireworksAI.

In [ ]:
##Construimos el prompt
PROMPT_EXTRACCION = """Actúa como un algoritmo avanzado de extracción de datos. Dado un JSON con información sobre un anuncio de vivienda y extrae datos estructurados, resolviendo posibles inconsistencias según reglas de negocio específicas.
    
    Deberás devolver ÚNICAMENTE un objeto JSON con las siguientes claves (en el orden indicado) y reglas:
    - "es_solar": (entero) 1 si se trata de un solar en el que no hay vivienda como tal o la casa está en ruinas, 0 en caso se haber un inmueble habitable.
    - "tipo_inmueble": (string) Clasifica si el inmueble es una "casa", "chalet", "apartamento", "piso", "duplex", "estudio", etc. Devuelve null si no se puede determinar.
    - "metros_cuadrados_construidos": (entero) Metros cuadrados construidos. IMPORTANTE: Si solo se indica una cantidad de metros (sin indicar de que tipo son) se asume que son construidos.
    - "metros_cuadrados_utiles": (entero) Metros cuadrados habitables o útiles. IMPORTANTE: NO extraer los metros cuadrados construidos. Si no se indica devuelve null.
    - "numero_plantas": (entero) Número de plantas que tiene el inmueble (útil para casas, chalets o dúplex). Devuelve 1 si no aplica o no se menciona.
    - "planta_piso": (entero) La altura o piso en el que se encuentra el inmueble (ej: 1 para primero, 2 para segundo, 3 para tercero). Devuelve 0 si es planta baja o bajo. Devuelve null si no se especifica.
    - "es_bajo": (entero) 1 si es una planta baja o un bajo, 0 si no.
    - "es_atico": (entero) 1 si es un ático, 0 si no.
    - "tiene_ascensor": (entero) 1 si tiene ascensor en funcionamiento. REGLA: Si se menciona que tiene ascensor pero que no está operativo, está averiado o hay que repararlo, devuelve 0.
    - "habitaciones": (entero) Número de habitaciones reales actuales. REGLA DE INCONSISTENCIA: Si hay discrepancias o se menciona "tiene X habitaciones pero pueden sacarse Y habitaciones", quédate siempre con las características más claras y actuales. La respuesta debe ser la cantidad actual (X), sin contemplar posibles obras, reformas o modificaciones del estilo.
    - "numero_baños": (entero) Número de baños y/o aseos que tiene el inmueble. Devuelve 1 si no se menciona.
    - "tiene_terraza": (entero) 1 si menciona que tiene terraza, 0 si no.
    - "tiene_balcon": (entero) 1 si menciona que tiene balcón, 0 si no.
    - "tiene_garaje": (entero) 0 si no tiene o se indica que se puede negociar aparte o que  no se incluye en el precio, 1 si tiene garaje o plaza de aparcamiento incluido en  el precio.
    - "tiene_trastero": (entero) 1 si tiene trastero, 0 si no.
    - "tiene_aire_acondicionado": (entero) 1 si menciona específicamente aire acondicionado o climatización, 0 si no.
    - "tiene_calefaccion": (entero) 1 si menciona específicamente calefacción o climatización, 0 si no.
    - "tiene_piscina": (entero) 1 si tiene piscina, 0 si no.
    - "tiene_jardin": (entero) 1 si tiene jardín, 0 si no.
    - "estado_ocupacion": (string) Clasifica el estado del inmueble estrictamente en uno de estos valores: "vacio", "arrendado", o "posible_ocupacion". REGLAS IMPORTANTES:
      1. Si en la descripción se indica que el inmueble "no está en propiedad", "no se puede acceder a él", "no se puede visitar" o similar, asume que está ocupado y el valor DEBE ser "posible_ocupacion".
      2. Si el inmueble tiene muy pocas fotos (3 o menos fotos) o directamente no tiene fotos, asume que el vendedor no ha podido acceder a tomar fotos y el valor DEBE ser "posible_ocupacion".
      3. Si se indica que está "alquilado", "arrendado", "con inquilino" o "en rentabilidad", el valor es "arrendado".
      4. Si no se cumple nada de lo anterior y parece estar disponible para uso, el valor es "vacio".
    - "necesita_reforma": (entero) 1 si indica que es para reformar, 0 si está en buen estado u obra nueva.
    - "tipo_vendedor": (string) "particular" si se trata de un vendedor particular, "agencia" si se trata de una agencia inmobiliara, vacío si no somos posibles de identificarlo.
    
    Información del anuncio:
    {json_escrapeado}"""

#Declaramos las columnas a recuperar, indicadas en el PROMPT
EXPECTED_KEYS=["es_solar", "tipo_inmueble", "metros_cuadrados_construidos", "metros_cuadrados_utiles", 
                        "numero_plantas", "planta_piso", "es_bajo", "es_atico", "tiene_ascensor", "habitaciones", 
                        "numero_baños", "tiene_terraza", "tiene_balcon", "tiene_garaje", "tiene_trastero", 
                        "tiene_aire_acondicionado", "tiene_calefaccion", "tiene_piscina", "tiene_jardin", 
                        "estado_ocupacion", "necesita_reforma", "tipo_vendedor"]

#Definimos el esquema que va a tener la respuesta en spark
EXPECTED_ESCHEMA=StructType([
    StructField("es_solar", IntegerType(), True),
    StructField("tipo_inmueble", StringType(), True),
    StructField("metros_cuadrados_construidos", IntegerType(), True),
    StructField("metros_cuadrados_utiles", IntegerType(), True),
    StructField("numero_plantas", IntegerType(), True),
    StructField("planta_piso", IntegerType(), True),
    StructField("es_bajo", IntegerType(), True),
    StructField("es_atico", IntegerType(), True),
    StructField("tiene_ascensor", IntegerType(), True),
    StructField("habitaciones", IntegerType(), True),
    StructField("numero_baños", IntegerType(), True),
    StructField("tiene_terraza", IntegerType(), True),
    StructField("tiene_balcon", IntegerType(), True),
    StructField("tiene_garaje", IntegerType(), True),
    StructField("tiene_trastero", IntegerType(), True),
    StructField("tiene_aire_acondicionado", IntegerType(), True),
    StructField("tiene_calefaccion", IntegerType(), True),
    StructField("tiene_piscina", IntegerType(), True),
    StructField("tiene_jardin", IntegerType(), True),
    StructField("estado_ocupacion", StringType(), True),
    StructField("necesita_reforma", IntegerType(), True),
    StructField("tipo_vendedor", StringType(), True)
])

#Definimos la función para recuperar la respuesta del modelo para el prompt
def _get_json_reponse_fireworks_ai(datos_crudos_json_str):
    try:
        prompt=PROMPT_EXTRACCION.format(json_escrapeado=datos_crudos_json_str)
        expected_keys=EXPECTED_KEYS
      
        raw_content=query_json_to_fireworks_ai(prompt=prompt, expected_keys=expected_keys)
        
        return raw_content
    except:
        return None

#Definimos la función para recuperar, toma la anterior pero lanza un reintento
def get_json_reponse_fireworks_ai(datos_crudos_json_str):
    response=_get_json_reponse_fireworks_ai(datos_crudos_json_str)
    
    if response is not None:
        return response

    #En otro caso relanzamos
    return _get_json_reponse_fireworks_ai(datos_crudos_json_str)   

## 1. Lectura de datos
Creamos una función que lea los datos que aún no han sido procesados, esto lo hace usando una columna de metadatos que lo indica, y tomando la lectura por baches, además de leer esa información crea una columna cuyo contenido es un json (string) con ciertas columnas seleccionadas, el resultado de esta función será necesaria para la aplicación de la función definida antes.

In [ ]:
#Funcion de lectura de datos (pendientes y por baches)
def read_data_to_process(provincia, spark, bach_size=20):    
    df_detalle_ofertas=(tfm_utils.read_from_delta(spark, DETALLE_TABLE)
                        .filter(F.col('provincia') == provincia)
                        .filter(F.col('_scraped_status').like('ok'))
                        .filter((F.col('_processed_ai').isNull()) | (F.col('_processed_ai')!=F.lit(1)))
                        .limit(bach_size)
                    )

    # Columnas que formarán el contexto del prompt
    prompt_columns = ["caracteristicas_basicas", "cert_energetico", "description",
        "equipamiento", "gastos_comunidad_raw", "has_price_drop", "num_fotos",
        "price_drop_pct", "title", "updated"]
    
    #Columnas que no aportan o que descartamos de cara a la siguiente etapa
    columns_to_drop=["photo_urls" ,"location" ,"plan_urls" ,"_error_msg" 
                     ,"_is_deactivated" ,"_scraped_at" ,"_scraped_status"]

    df_to_prompt=(df_detalle_ofertas
                    .withColumn("id_url", F.regexp_extract(F.col("url"), r"(\d+)", 1))
                    .withColumn("to_prompt", F.to_json(F.struct(*[F.col(c) for c in prompt_columns])))
                    .withColumn("to_prompt", F.regexp_replace(F.col("to_prompt"), r'\\"', '"'))
                    .drop(*prompt_columns, *columns_to_drop)
                )
    
    data_to_process=df_to_prompt.limit(1).count() #1 si hay lineas a procesar, 0 si no queda nada
    
    return data_to_process, df_to_prompt

## 2. Recuperación de información usando IA
Creamos una función que dado un dataframe de Spark con la columna de prompt antes construida, pasa dicho dataframe a Pandas, le aplica la función para recuperar el json con el resultado del modelo, y además lo aplana, recuperando cada una de las claves que recuperamos con el prompt. Además devolvemos los datos procesados y aquellos que no, así podremos cambiar el metadato de haber sido o no procesado de cara a una siguiente ejecución.

In [ ]:
def get_info_ai_fireworks(spark, df_with_promt_col):
    #pasamos el dataframe a pandas para simplificar la operación
    df_to_process_pd=df_with_promt_col.toPandas()

    #Le pasamos la columna de prompt al modelo de Fireworks
    df_to_process_pd["raw_content_FireworksAI"]=df_to_process_pd["to_prompt"].parallel_apply(get_json_reponse_fireworks_ai)

    #Volvemos a pasar a PySpark
    df_FireworksAI_processed=spark.createDataFrame(df_to_process_pd)
    
    #nos quedamos con los registros procesados con error o fallo
    df_not_processed=(df_FireworksAI_processed
                        .filter(F.col("raw_content_FireworksAI").isNull())
                        .select('provincia', 'municipio', 'url')
                     )
    
    #nos quedamos con los registros procesados correctamente
    df_processed=df_FireworksAI_processed.filter(F.col("raw_content_FireworksAI").isNotNull())

    #Damos forma a los registros recuperados  
    df_processed=(df_processed
                    #Aplicamos el formato a la columna recuperada
                    .withColumn(f"final_json_FireworksAI", F.from_json(F.col("raw_content_FireworksAI")
                                                                        ,EXPECTED_ESCHEMA
                                                                       )
                                )
                    #En caso de obtener una cadena que no encaja con el esquema from_json devuelve null
                    .filter(F.col("final_json_FireworksAI").isNotNull())
                    .drop('id_url', 'to_prompt', 'raw_content_FireworksAI')
                 )
    #Aplanamos extrallendo todas las columnas de la estructura
    for clave in EXPECTED_KEYS:
        df_processed=(df_processed
                        .withColumn(f"{clave}", F.col(f"final_json_FireworksAI.{clave}"))
                    )

    df_processed=df_processed.drop("final_json_FireworksAI")
    
    #Devolvemos los datos procesados y los no procesados
    return df_processed, df_not_processed

## 3. Escritura de los datos recuperados
Definimos primero una función con la cual guardamos los datos en la capa silver, y además otra con la actualizamos el metadato he haber sido procesado en la tabla de donde tomamos los datos.

In [ ]:
#Definimos la función con la cual persistir la información procesada en la tabla de silver
def update_silver_table(spark, df_new_data):
    silver_table="silver.idealista.detalle_propiedades"
    silver_table_path=tfm_utils.full_table_path(silver_table)
    partition_cols=['provincia', 'municipio']

    try:
        #Hacemos un doble TRY-EXCEPT para priorizar la primera escritura si no existe la tabla
        try:
            #Si la tabla no existe fallará el select siguiente
            spark.sql(f"SELECT 1 AS example FROM delta.`{silver_table_path}` LIMIT 1")
        except:
            #Si no falla hacemos un escribimos la nueva información por primera vez
            (df_new_data.write
                .format("delta")
                .mode("append")
                .partitionBy(*partition_cols)
                .save(silver_table_path)
            )
            
    except:
        #Si la table existe hacemos un merge por si los datos que procesamos
        #se han guardado con anterioridad en una ejecución fallida
        dt_silver = tfm_utils.get_delta_table(spark, silver_table_path)

        partition_cols=['provincia', 'municipio', 'url']
        sub_conditions = [f"target.{col}=source.{col}" for col in partition_cols]
        condition=" AND ".join(sub_conditions)
        
        (dt_silver.alias('target')
         .merge(df_new_data.alias('source'), condition)
         .whenNotMatchedInsertAll()#Metemos solo lo nuevo, asumimos que el resultado del modelo no cambia
         .execute()        
        )
   
    print(f"Datos guardados en {silver_table}")
    
#Definimos la función con la cual actualizar el metadato de procesamiento en raw
def update_processed_ai(spark, df_processed, df_not_processed):
    dt_detalle = tfm_utils.get_delta_table(spark, DETALLE_TABLE)
    
    partition_cols=['provincia', 'municipio', 'url']
    sub_conditions = [f"target.{col}=source.{col}" for col in partition_cols]
    condition=" AND ".join(sub_conditions)

    #Mergeamos los procesados
    (dt_detalle.alias('target')
         .merge(df_processed.alias('source'), condition)
         .whenMatchedUpdate(set={"_processed_ai":"1"})#los marcamos como procesados
         .execute()        
    )

    #Mergeamos los NO procesados, esto es simplemente para saber que han pasado por el modelo
    (dt_detalle.alias('target')
         .merge(df_not_processed.alias('source'), condition)
         .whenMatchedUpdate(set={"_processed_ai":"0"})#los marcamos como procesados
         .execute()        
    )

## 4. Orquestación y puesta en marcha del trabajo

In [ ]:
provincia  = 'murcia'

#Creamos una sesión de Spark
spark = tfm_utils.get_spark_session(f'Idealista_Detalle_{provincia}')

#Se crean variables para poder lanzar por baches
bach_size=100
max_iterations=25 #control val
iteration=0


total_datos_procesados=0
run=True

while run and iteration<max_iterations:
    #Verificamos si hay datos a procesar y los recuperamos
    data_to_process, df_with_prompt=read_data_to_process(provincia, spark, bach_size)

    #Si no quedan datos a procesar salimos del bucle
    if data_to_process==0:
        run=False
        break
    #Aumentamos la variable de iteración antes de continuar
    iteration+=1

    #Extraemos los campos con IA donde se pueda
    df_processed, df_not_processed=get_info_ai_fireworks(spark,df_with_prompt)

    if df_processed.limit(1).count()>0:
        df_processed=(df_processed
                    .withColumn("df_new_data",F.lit(1))    
                )

        #Guardamos la información procesada
        update_silver_table(spark, df_processed)
    
        #Cambiamos los metadatos para los datos procesados y no procesados
        update_processed_ai(spark, df_processed, df_not_processed)
    
        total_datos_procesados+=df_processed.count()
        
        print(f"Datos recuperados {total_datos_procesados}")

Datos guardados en silver.idealista.detalle_propiedades
Datos recuperados 6
